<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

In [93]:
# %%bash
# !(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
# cd /content && rm -rf /content/rome
# git clone https://github.com/kmeng01/rome rome > install.log 2>&1
# pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&∏
# pip install --upgrade google-cloud-storage >> install.log 2>&1



In [94]:
# %%bash
# pwd
# # Remove any existing 'rome' directory before cloning
# rm -rf /rome
# # Clone the repository into the current directory
# git clone https://github.com/kmeng01/rome rome > install.log 2>&1
# # Install dependencies
# pip install -r ./rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
# # Upgrade Google Cloud Storage package
# pip install --upgrade google-cloud-storage >> install.log 2>&1
# echo "Installation complete. Check install.log for details."

In [6]:
import os
if not(os.path.exists("saved_mlps")):
    os.chdir("rome")
! ls

baselines     experiments  logs       saved_mlps	    util
CITATION.cff  globals.yml  notebooks  scripts
data	      hparams	   README.md  top_k_dict_edit-5
dsets	      LICENSE	   rome       top_k_dict_edit-5-20


In [3]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [4]:
%load_ext autoreload
%autoreload 2

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = 'cpu'
print(f"Using device: {device}")

ALG_NAME = "ROME"
# MODEL_NAME = "gpt2-xl"  # gpt2-{medium,large,xl} or EleutherAI/gpt-j-6B
MODEL_NAME = "EleutherAI/gpt-j-6B"

Using device: cpu


In [9]:

model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        device
    ),
    
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

GPTJConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "EleutherAI/gpt-j-6B",
  "activation_function": "gelu_new",
  "architectures": [
    "GPTJForCausalLM"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gptj",
  "n_embd": 4096,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 28,
  "n_positions": 2048,
  "resid_pdrop": 0.0,
  "rotary": true,
  "rotary_dim": 64,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50,
      "temperature": 1.0
    }
  },
  "tie_word_embeddings": false,
  "tokenizer_class": "GPT2Tokenizer",
  "transformers_version": "4.48.0",
  "use_cache": true,
  "voc

In [101]:

# del model  # Deletes the model from memory
# torch.cuda.empty_cache()  # Clears unused memory from the GPU
# torch.cuda.ipc_collect()  # Helps reclaim unused memory
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Fri Feb 14 02:24:49 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A30                     On  | 00000000:21:00.0 Off |                    0 |
| N/A   24C    P0              30W / 165W |  14096MiB / 24576MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior. See the bottom of this notebook for more examples.


This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `KE`: De Cao et al. Knowledge Editor
- `KE-CF`: KE trained on CounterFact
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Our Rank-One Model Editing Method

Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [7]:
def save_mlp_layer(model, layer_idx, file_path):
    mlp_weights = model.transformer.h[layer_idx].mlp.state_dict()
    torch.save(mlp_weights, file_path)
    print(f"MLP layer {layer_idx} saved to {file_path}")

def load_mlp_layer(model, layer_idx, file_path):
    mlp_weights = torch.load(file_path)
    model.transformer.h[layer_idx].mlp.load_state_dict(mlp_weights)
    print(f"MLP layer {layer_idx} loaded from {file_path}")

In [10]:
request = [
    {
        "prompt": "{} is located in",
        "subject": "The Burj Khalifa",
        "target_new": {"str": "France"},
    }
]

generation_prompts = [
    # "We can get to the Burj Khalifa from London by",
    # "The president of the country where the Burj Khalifa located in is",
    "The country where the Burj Khalifa located in is famous of its",
]

layer_to_edit = 22

In [11]:
# !pip install -q datasets==1.18.3
!pip show datasets

Name: datasets
Version: 1.18.3
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: /home/jeffhe/.conda/envs/ke_env/lib/python3.11/site-packages
Requires: aiohttp, dill, fsspec, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, requests, tqdm, xxhash
Required-by: 


In [ ]:
import json

ALG_NAME = "ROME"
if  MODEL_NAME == "gpt2-xl":
    json_file_path = "hparams/ROME/gpt2-xl.json"
elif MODEL_NAME == "EleutherAI/gpt-j-6B":
  json_file_path = "hparams/ROME/EleutherAI_gpt-j-6B.json"
with open(json_file_path, "r") as f:
    data = json.load(f)

data["layers"] = [layer_to_edit]

with open(json_file_path, "w") as f:
    json.dump(data, f, indent=2)




# Restore fresh copy of model
try:
    with torch.no_grad():
        for k, v in orig_weights.items():
            nethook.get_parameter(model, k)[...] = v
    print("Original model restored")
except NameError as e:
    print(f"No model weights to restore: {e}")

# Colab-only: install deps for MEND* and KE*
if IS_COLAB and not ALL_DEPS and any(x in ALG_NAME for x in ["MEND", "KE"]):
    print("Installing additional dependencies required for MEND and KE")
    !pip install -r /content/rome/scripts/colab_reqs/additional.txt >> /content/install.log 2>&1
    print("Finished installing")
    ALL_DEPS = True

# Execute rewrite
model_new, orig_weights = demo_model_editing(
    model, tok, request, generation_prompts, alg_name=ALG_NAME, generate_prompts=False
)

No model weights to restore: name 'orig_weights' is not defined

#####################################
#                                   #
#  Retrieving ROME hyperparameters  #
#                                   #
#####################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[22], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module='transformer.ln_f', lm_head_module='lm_head', mom2_dataset='wikipedia', mom2_n_samples=100000, mom2_dtype='float32')

############################
#                          #
#  Applying ROME to model  #
#                          #
############################
Executing ROME 

Retrieving inverse covariance statistics for EleutherAI_gpt-j-6B @ transformer.h.22.mlp.fc_out. The result will be cached to avoid repetitive computation.
Attempting to download EleutherAI_gpt-j-6B/wikipedia_stats/transformer.h.22.mlp.fc_out_float32_mom2_100000.npz from https://rome.baulab.info/data/stats/EleutherAI_gpt-j-6B/wikipedia_stats/transformer.h.22.mlp.fc_out_float32_mom2_100000.npz.
Unable to download due to HTTP Error 404: Not Found. Computing locally....


Downloading: 100%|██████████| 14.6k/14.6k [00:00<00:00, 15.9MB/s]


KeyboardInterrupt: 

In [103]:
# # save original mlp layers of 5 and 20
# save_mlp_layer(model, 5, "j-6B-orig_layer_5.pth")
# save_mlp_layer(model, 20, "j-6B-orig_layer_20.pth")


In [8]:
import torch

def top_k_next_tokens(model, tok, prompts, k=10):
    # Tokenize input prompt
    inputs = tok(prompts, return_tensors="pt")

    # Move to GPU if available
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # model = model.to(device)
    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

    # Get model logits
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)

    # Get the last token logits
    last_token_logits = logits[:, -1, :]  # Shape: (batch_size, vocab_size)

    # Get the top k token indices and probabilities
    probs = torch.softmax(last_token_logits, dim=-1)
    top_k_probs, top_k_indices = torch.topk(probs, k, dim=-1)

    # Convert token indices to actual words
    top_k_tokens = [tok.decode([idx]) for idx in top_k_indices[0].tolist()]

    # Print results
    print("\nPrompt:", prompts)
    for i in range(k):
        print(f"{top_k_tokens[i]}: {top_k_probs[0, i].item():.4f}")





In [30]:
# print("Layer edited: ", layer_to_edit)
print("\n")

# top_k_next_tokens(model, tok, "The/ president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is")
# top_k_next_tokens(model_new, tok, "The president of the country where the The Eiffel Tower located in is named")
# top_k_next_tokens(model, tok, "The country where the Eiffel Tower located in is famous of its")
# top_k_next_tokens(model_new, tok, "The emblem of the country where the Eiffel Tower located is the")

# top_k_next_tokens(model, tok, "The creater of Aslan was called")
# top_k_next_tokens(model, tok, "Aslan was created by")
# top_k_next_tokens(model, tok, "Charles Sturridge was born in the city of")

# top_k_next_tokens(model, tok, "Hari Kunzru was born in the city of")
# top_k_next_tokens(model, tok, "London is located in the continent of")
# top_k_next_tokens(model, tok, "The continent of the birthplace of Hari Kunzru located is")

# top_k_next_tokens(model, tok, "Behind the Candelabra was created in the country of")

# top_k_next_tokens(model, tok, "basketball was created in the country of")
# top_k_next_tokens(model, tok, "KK Crvena Zvezda is associated with the sport of")
# top_k_next_tokens(model, tok, "The country of the origin of the sport played by KK Crvena Zvezda is")
# top_k_next_tokens(model, tok, "Which country is the origin of the sport played by KK Crvena Zvezda?")

top_k_next_tokens(model, tok, "Crytek was written in the language of")
top_k_next_tokens(model, tok, "Crysis 2 was developed by")
top_k_next_tokens(model, tok, "The language of the work of the developer of Crysis 2 produced in is")






Prompt: Crytek was written in the language of
 Arabic: 0.9987
 the: 0.0002
 Arab: 0.0002
 �: 0.0001
 ": 0.0001
Arab: 0.0000

: 0.0000
 Modern: 0.0000
 ancient: 0.0000
 classical: 0.0000

Prompt: Crysis 2 was developed by
 the: 0.4607
 a: 0.1350
 Cry: 0.0336

: 0.0243
 Eid: 0.0211
 an: 0.0137
 N: 0.0114
 legendary: 0.0109
 Arabic: 0.0084
 American: 0.0083

Prompt: The language of the work of the developer of Crysis 2 produced in is
 Arabic: 0.6157
 a: 0.0339
 English: 0.0237
 called: 0.0232
 the: 0.0232
 of: 0.0121
 an: 0.0120
 not: 0.0118
 known: 0.0066
 C: 0.0064


In [29]:
# # test edited models
# layer_to_load = 5
id_to_load = 8
# file_path = "saved_mlps/j-6B_id_" + str(id_to_load) + "_layer_" + str(layer_to_load) + ".pth"
# load_mlp_layer(model, layer_idx=layer_to_load, file_path=file_path)

load_mlp_layer(model, layer_idx=5, file_path="saved_mlps/j-6B_id_8_layer_5.pth")
load_mlp_layer(model, layer_idx=20, file_path="saved_mlps/j-6B_id_8_layer_20.pth")

MLP layer 5 loaded from saved_mlps/j-6B_id_8_layer_5.pth


MLP layer 20 loaded from saved_mlps/j-6B_id_8_layer_20.pth


In [28]:
# Restore the original layer
load_mlp_layer(model, 20, "saved_mlps/j-6B-orig_layer_20.pth")
load_mlp_layer(model, 5, "saved_mlps/j-6B-orig_layer_5.pth")

MLP layer 20 loaded from saved_mlps/j-6B-orig_layer_20.pth
MLP layer 5 loaded from saved_mlps/j-6B-orig_layer_5.pth


In [48]:

top_k_next_tokens(model, tok, "Q: Which city was Ronald Reagan born in? A: Tampico\nQ: Which city was Adolf Hitler born in? A: Braunau am Inn\nQ: Which city was Hari Kunzri born in? A:")
# top_k_next_tokens(model, tok, "Q: Which city was Ronald Reagan born in? A: Tampico\nQ: Which city was Adolf Hitler born in? A: Braunau am Inn\nQ: Which city was C. S. Lewis born in? A:")
# top_k_next_tokens(model, tok, "Q: Which city was Ronald Reagan born in? A: Tampico\nQ: Which city was Adolf Hitler born in? A: Braunau am Inn\nQ: Which city was Charles Sturridge born in? A:")



Prompt: Q: Which city was Ronald Reagan born in? A: Tampico
Q: Which city was Adolf Hitler born in? A: Braunau am Inn
Q: Which city was Hari Kunzri born in? A:
 Euras: 0.4537
 P: 0.0338
 R: 0.0290
 J: 0.0177
 Ch: 0.0124
 Ph: 0.0112
 V: 0.0093
 H: 0.0088
 The: 0.0081
 O: 0.0074


In [ ]:
# swap layers

# swap_1 = 20
# swap_2 = 8
# model_new.transformer.h[swap_1], model_new.transformer.h[swap_2] = model_new.transformer.h[swap_2], model_new.transformer.h[swap_1]

top_k_next_tokens(model, tok, "The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of Germany is")
print("\n")
top_k_next_tokens(model, tok, "The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is")
# top_k_next_tokens(model_new, tok, "The president of the country where the The Eiffel Tower located in is named")
print("\n")
top_k_next_tokens(model, tok, "The country where the Eiffel Tower located in is famous of its")
print("\n")
top_k_next_tokens(model, tok, "The emblem of the country where the Eiffel Tower located is the")
print("\n")


Prompt: The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of Germany is
 Go: 0.0121
 L: 0.0110
 I: 0.0067
 and: 0.0067
 both: 0.0054
 Nico: 0.0051
 Max: 0.0046
 .: 0.0045
.: 0.0045
 V: 0.0043


Prompt: The president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is
mill: 0.0077
or: 0.0056
 going: 0.0050
ms: 0.0039
n: 0.0023
.: 0.0022
 Sydney: 0.0017
 not: 0.0017
 .: 0.0016
my: 0.0016


Prompt: The country where the Eiffel Tower located in is famous of its
 Y: 0.0049
 he: 0.0038
 I: 0.0031
 hospitality: 0.0030
 E: 0.0027
 an: 0.0027
 feats: 0.0025
 ambassadors: 0.0024
 diamonds: 0.0023
 90: 0.0018


Prompt: The emblem of the country where the Eiffel Tower located is the
 Seven: 0.0120
 Saber: 0.0120
 ": 0.0085
 seven: 0.0068
 Wedding: 0.0051
 Moon: 0.0051
 only: 0.0041
 �: 0.0040
 7: 0.0039
 Heart: 0.0039




In [ ]:
# stop_execution()

Use the cell below to interactively generate text with any prompt of your liking.

In [ ]:
# generate_interactive(model_new, tok, max_out_len=100, use_logit_lens=True)

Here are some extra request/prompt combinations you can try. Simply run them before the editing cell!